# 05 Chart and Signal Scanner - MA Cross

Scanner va chart chi de quan sat tin hieu. Khong dung chart dep de ket luan hieu qua; ket luan phai di qua backtest cost-aware.

In [ ]:
import sys
from pathlib import Path

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'core_python' / 'shared').exists():
            return p
    raise RuntimeError('Cannot find SEN05 repo root')

ROOT = find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('ROOT =', ROOT)

In [ ]:
import pandas as pd
from IPython.display import display

from core_python.strategies.ma_cross.params import SYMBOLS, TIMEFRAMES, get_symbol_params
from core_python.strategies.ma_cross.model import add_ma_cross_indicators, detect_ma_cross_signals, session_mask
from core_python.strategies.ma_cross.research_utils import configure_notebook, show_run_config, show_strategy_summary
from core_python.strategies.ma_cross.symbol.backtest import load_backtest_data

configure_notebook()
show_strategy_summary()

In [ ]:
RUN_CONFIG = {
    'symbols': ['US30', 'NAS100', 'GOLD', 'EURUSD', 'BTCUSD'],
    'tf': 'M30',
    'max_bars': 3_000,
    'date_to': None,
    'indicator_overrides': {},
    'broker_profile': None,
}
show_run_config('MA Cross scanner config', RUN_CONFIG)

In [ ]:
rows = []
frames = {}
for symbol in RUN_CONFIG['symbols']:
    cfg = get_symbol_params(symbol, broker_profile=RUN_CONFIG['broker_profile'])
    raw = load_backtest_data(cfg['symbol_id'], tf=RUN_CONFIG['tf'], max_bars=RUN_CONFIG['max_bars'], date_to=RUN_CONFIG['date_to'])
    ind = add_ma_cross_indicators(raw, RUN_CONFIG['indicator_overrides'])
    sig = detect_ma_cross_signals(ind, session_mask(ind, cfg.get('session_hours_utc', [])), sym_key=symbol)
    frames[symbol] = sig
    last_signal = sig[sig['signal'] != 0].tail(1)
    rows.append({
        'symbol': symbol,
        'last_bar': sig.index[-1],
        'last_close': sig['close'].iloc[-1],
        'last_signal_time': last_signal.index[-1] if not last_signal.empty else None,
        'last_signal': int(last_signal['signal'].iloc[-1]) if not last_signal.empty else 0,
        'ma_gap_atr': sig['ma_gap_atr'].iloc[-1],
    })

scanner = pd.DataFrame(rows)
display(scanner)

chart_symbol = RUN_CONFIG['symbols'][0]
frames[chart_symbol][['close', 'fast_ma', 'slow_ma']].tail(300).plot(title=f'{chart_symbol} MA Cross visual check', figsize=(12, 4));